# Day 01 — Raw Read Quality Control with FastQC & MultiQC

**#30DaysOfBioinformatics | SubhadipJana1409**

---

## Research Context
Quality control is the most critical — and most overlooked — step in any NGS analysis. 
Andrews et al. (2010) **FastQC** remains the gold standard for per-base quality scores, GC content, 
adapter contamination, and duplication rate inspection. **MultiQC** aggregates reports across 
dozens of samples, surfacing batch-level anomalies invisible in individual sample QC.

## Dataset
| Field | Value |
|---|---|
| GEO Accession | GSE96960 |
| SRA Project | SRP100973 |
| Organism | *Homo sapiens* |
| Library | Paired-end, Illumina HiSeq 2500, 150 bp |
| Samples | 6 PBMC (3 Healthy Controls + 3 Septic Shock) |
| SRR IDs | SRR5223500–SRR5223505 |
| Reference | Liao et al. (2017) *JCI Insight* |

## Workflow Overview
```
Raw FASTQ → FastQC → MultiQC → Flag QC issues → fastp trim → Post-trim FastQC/MultiQC → Summary table
```

## Tools Required
```bash
conda create -n day01_qc -c bioconda -c conda-forge \
    fastqc multiqc fastp sra-tools pandas matplotlib seaborn jupyter
conda activate day01_qc
```

---
## 0. Environment Setup

In [ ]:
import os
import re
import subprocess
import json
from pathlib import Path
from glob import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# ── Palette ──────────────────────────────────────────────────────────────────
PASS_COLOR  = "#2ecc71"   # green
WARN_COLOR  = "#f39c12"   # amber
FAIL_COLOR  = "#e74c3c"   # red
HC_COLOR    = "#3498db"   # healthy
SS_COLOR    = "#e74c3c"   # septic

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "sans-serif"

# ── Directories ───────────────────────────────────────────────────────────────
BASE_DIR     = Path(".").resolve()
RAW_DIR      = BASE_DIR / "data" / "raw_fastq"
TRIM_DIR     = BASE_DIR / "data" / "trimmed_fastq"
QC_RAW_DIR   = BASE_DIR / "qc" / "raw"
QC_TRIM_DIR  = BASE_DIR / "qc" / "trimmed"
MULTIQC_RAW  = BASE_DIR / "qc" / "multiqc_raw"
MULTIQC_TRIM = BASE_DIR / "qc" / "multiqc_trimmed"
FASTP_DIR    = BASE_DIR / "qc" / "fastp_reports"
RESULTS_DIR  = BASE_DIR / "results"
FIG_DIR      = BASE_DIR / "figures"

for d in [RAW_DIR, TRIM_DIR, QC_RAW_DIR, QC_TRIM_DIR,
          MULTIQC_RAW, MULTIQC_TRIM, FASTP_DIR, RESULTS_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Sample metadata ──────────────────────────────────────────────────────────
SAMPLES = {
    "SRR5223500": {"group": "HC", "label": "HC_1"},
    "SRR5223501": {"group": "HC", "label": "HC_2"},
    "SRR5223502": {"group": "HC", "label": "HC_3"},
    "SRR5223503": {"group": "SS", "label": "SS_1"},
    "SRR5223504": {"group": "SS", "label": "SS_2"},
    "SRR5223505": {"group": "SS", "label": "SS_3"},
}

THREADS = 8  # adjust to your machine

print("✓ Setup complete")
print(f"  Base directory : {BASE_DIR}")
print(f"  Threads        : {THREADS}")
print(f"  Samples        : {len(SAMPLES)} ({sum(1 for v in SAMPLES.values() if v['group']=='HC')} HC + {sum(1 for v in SAMPLES.values() if v['group']=='SS')} SS)")

---
## Step 1 — FastQC on Raw FASTQ Files
Run FastQC on all 12 files (R1 + R2 for 6 samples). Captures per-base quality, GC distribution, adapter content, and k-mer profiles.

In [ ]:
def run_fastqc(fastq_dir: Path, output_dir: Path, threads: int = 8) -> None:
    """Run FastQC on all .fastq.gz files in fastq_dir."""
    fastq_files = sorted(fastq_dir.glob("*.fastq.gz"))
    if not fastq_files:
        raise FileNotFoundError(f"No .fastq.gz files found in {fastq_dir}")

    print(f"Running FastQC on {len(fastq_files)} files → {output_dir}")
    cmd = [
        "fastqc",
        "--threads", str(threads),
        "--outdir", str(output_dir),
        "--extract",          # unzip HTML for parsing later
        *[str(f) for f in fastq_files],
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError("FastQC failed")
    print(f"  ✓ FastQC complete — {len(fastq_files)} reports written")


# Run FastQC on raw reads
run_fastqc(RAW_DIR, QC_RAW_DIR, THREADS)

---
## Step 2 — Aggregate with MultiQC

In [ ]:
def run_multiqc(search_dir: Path, output_dir: Path, report_name: str = "multiqc_report") -> None:
    """Aggregate FastQC/fastp reports with MultiQC."""
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Running MultiQC on {search_dir} → {output_dir}")
    cmd = [
        "multiqc",
        str(search_dir),
        "--outdir", str(output_dir),
        "--filename", report_name,
        "--force",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print("STDERR:", result.stderr[-2000:])
        raise RuntimeError("MultiQC failed")
    html = output_dir / f"{report_name}.html"
    print(f"  ✓ MultiQC report: {html}")


# Run MultiQC on raw FastQC outputs
run_multiqc(QC_RAW_DIR, MULTIQC_RAW, report_name="multiqc_raw")

---
## Step 3 — Parse FastQC Reports & Flag Problem Samples
Flag samples with:
- Q < 20 tails (per-base sequence quality FAIL)
- Adapter contamination > 5%
- GC content deviating > 10% from expected (~50% for human PBMC)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Parse FastQC summary.txt files
# Each summary.txt contains: PASS/WARN/FAIL \t MODULE \t FILENAME
# ─────────────────────────────────────────────────────────────────────────────

def parse_fastqc_summaries(qc_dir: Path) -> pd.DataFrame:
    """Parse all FastQC summary.txt files into a tidy DataFrame."""
    records = []
    for summary_file in sorted(qc_dir.rglob("summary.txt")):
        # Extract SRR ID and read pair from the parent directory name
        # e.g.  SRR5223500_1_fastqc/summary.txt
        dir_name = summary_file.parent.name          # SRR5223500_1_fastqc
        # strip trailing _fastqc
        base = dir_name.replace("_fastqc", "")
        # last token is read (1 or 2)
        parts = base.rsplit("_", 1)
        srr_id = parts[0]
        read   = f"R{parts[1]}" if len(parts) == 2 else "R1"

        with open(summary_file) as fh:
            for line in fh:
                status, module, _ = line.strip().split("\t")
                records.append({
                    "sample_id" : srr_id,
                    "group"     : SAMPLES.get(srr_id, {}).get("group", "?"),
                    "label"     : SAMPLES.get(srr_id, {}).get("label", srr_id),
                    "read"      : read,
                    "module"    : module,
                    "status"    : status,
                })
    return pd.DataFrame(records)


def parse_fastqc_data(qc_dir: Path) -> dict:
    """
    Extract numeric metrics from fastqc_data.txt:
    - %GC, total sequences, sequence length
    - Max adapter content %
    - Minimum per-base quality (Q20 threshold)
    """
    metrics = []
    for data_file in sorted(qc_dir.rglob("fastqc_data.txt")):
        dir_name = data_file.parent.name.replace("_fastqc", "")
        parts    = dir_name.rsplit("_", 1)
        srr_id   = parts[0]
        read     = f"R{parts[1]}" if len(parts) == 2 else "R1"

        gc_pct = total_seq = seq_len = min_q = max_adapter = None
        in_adapter_section = False
        adapter_values = []
        in_pbsq_section = False
        pbsq_min_q = []

        with open(data_file) as fh:
            for line in fh:
                line = line.strip()

                # Basic stats
                if line.startswith("%GC"):
                    gc_pct = float(line.split("\t")[1])
                elif line.startswith("Total Sequences"):
                    total_seq = int(line.split("\t")[1])
                elif line.startswith("Sequence length"):
                    seq_len = line.split("\t")[1]   # e.g. "150" or "35-150"

                # Per-base quality section — capture Mean column
                if line.startswith(">>Per base sequence quality"):
                    in_pbsq_section = True; continue
                if in_pbsq_section:
                    if line.startswith(">>END_MODULE"):
                        in_pbsq_section = False
                    elif not line.startswith("#"):
                        parts_q = line.split("\t")
                        try:
                            pbsq_min_q.append(float(parts_q[1]))  # Mean quality
                        except (IndexError, ValueError):
                            pass

                # Adapter content section — capture max % across all positions
                if line.startswith(">>Adapter Content"):
                    in_adapter_section = True; continue
                if in_adapter_section:
                    if line.startswith(">>END_MODULE"):
                        in_adapter_section = False
                    elif not line.startswith("#"):
                        vals = line.split("\t")[1:]  # skip position column
                        try:
                            adapter_values.extend([float(v) for v in vals])
                        except ValueError:
                            pass

        min_q       = min(pbsq_min_q)  if pbsq_min_q      else None
        max_adapter = max(adapter_values) if adapter_values else None

        metrics.append({
            "sample_id"     : srr_id,
            "group"         : SAMPLES.get(srr_id, {}).get("group", "?"),
            "label"         : SAMPLES.get(srr_id, {}).get("label", srr_id),
            "read"          : read,
            "total_reads_M" : round(total_seq / 1e6, 2) if total_seq else None,
            "seq_length"    : seq_len,
            "gc_pct"        : gc_pct,
            "min_mean_q"    : round(min_q, 2) if min_q else None,
            "max_adapter_pct": round(max_adapter, 2) if max_adapter else None,
        })
    return pd.DataFrame(metrics)


# Parse summaries
df_summary_raw = parse_fastqc_summaries(QC_RAW_DIR)
df_metrics_raw = parse_fastqc_data(QC_RAW_DIR)

print("FastQC summary table (raw):")
print(df_metrics_raw.to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Flag samples based on thresholds from the roadmap
# ─────────────────────────────────────────────────────────────────────────────
EXPECTED_GC   = 50.0   # expected GC% for human
GC_DEVIATION  = 10.0   # flag if |GC - expected| > 10%
ADAPTER_THRESH = 5.0   # flag if max adapter % > 5
MIN_Q_THRESH  = 20.0   # flag if any base-position mean Q < 20

def apply_qc_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["flag_low_quality"]  = df["min_mean_q"].apply(
        lambda q: "FAIL" if q is not None and q < MIN_Q_THRESH else "PASS")
    df["flag_adapter"]      = df["max_adapter_pct"].apply(
        lambda a: "FAIL" if a is not None and a > ADAPTER_THRESH else "PASS")
    df["flag_gc_bias"]      = df["gc_pct"].apply(
        lambda g: "FAIL" if g is not None and abs(g - EXPECTED_GC) > GC_DEVIATION else "PASS")
    df["overall_flag"]      = df.apply(
        lambda r: "FAIL" if "FAIL" in (r.flag_low_quality, r.flag_adapter, r.flag_gc_bias) else "PASS",
        axis=1)
    return df

df_flags_raw = apply_qc_flags(df_metrics_raw)

# Display flag table
flag_cols = ["label", "read", "group", "total_reads_M", "gc_pct",
             "min_mean_q", "max_adapter_pct",
             "flag_low_quality", "flag_adapter", "flag_gc_bias", "overall_flag"]
print("\nQC Flag Table (Raw Reads):")
print(df_flags_raw[flag_cols].to_string(index=False))

# Save
df_flags_raw[flag_cols].to_csv(RESULTS_DIR / "qc_flags_raw.csv", index=False)
print(f"\n✓ Saved: {RESULTS_DIR / 'qc_flags_raw.csv'}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 1 – FastQC Status Heatmap (all modules, all samples)
# ─────────────────────────────────────────────────────────────────────────────
STATUS_MAP = {"PASS": 0, "WARN": 1, "FAIL": 2}
COLOR_MAP  = {0: PASS_COLOR, 1: WARN_COLOR, 2: FAIL_COLOR}

def plot_fastqc_heatmap(df_summary: pd.DataFrame, title: str, save_path: Path) -> None:
    # Pivot: rows = module, columns = sample+read
    df_summary = df_summary.copy()
    df_summary["col"] = df_summary["label"] + "_" + df_summary["read"]
    pivot = df_summary.pivot_table(
        index="module", columns="col", values="status",
        aggfunc="first"
    )
    numeric = pivot.replace(STATUS_MAP)

    cmap = plt.cm.colors.ListedColormap([PASS_COLOR, WARN_COLOR, FAIL_COLOR])

    fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns) * 0.8), max(6, len(pivot) * 0.5)))
    im = ax.imshow(numeric.values.astype(float), cmap=cmap, vmin=0, vmax=2, aspect="auto")

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)

    # Add text annotations
    status_text = pivot.values
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = status_text[i, j] if status_text[i, j] else "?"
            ax.text(j, i, val[0], ha="center", va="center", fontsize=7,
                    color="white" if val == "FAIL" else "black")

    # Legend
    patches = [mpatches.Patch(facecolor=PASS_COLOR, label="PASS"),
               mpatches.Patch(facecolor=WARN_COLOR, label="WARN"),
               mpatches.Patch(facecolor=FAIL_COLOR, label="FAIL")]
    ax.legend(handles=patches, loc="upper right", bbox_to_anchor=(1.15, 1), fontsize=8)

    ax.set_title(title, fontsize=12, fontweight="bold", pad=12)
    plt.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  ✓ Figure saved: {save_path}")


plot_fastqc_heatmap(
    df_summary_raw,
    title="FastQC Module Status — Raw Reads (GSE96960 PBMC)",
    save_path=FIG_DIR / "fig1_fastqc_heatmap_raw.png"
)

---
## Step 4 — fastp Adapter Trimming & Quality Filtering
Parameters:
- Adapter auto-detection (fastp default)
- Sliding window quality trimming: Q20, window size 4
- Minimum read length: 50 bp after trimming
- Polyg tail trimming (common in Illumina NextSeq/NovaSeq data)

In [ ]:
def run_fastp(srr_id: str, raw_dir: Path, out_dir: Path,
              report_dir: Path, threads: int = 8) -> dict:
    """
    Run fastp on a paired-end sample.
    Returns a dict of trimming metrics parsed from the JSON report.
    """
    r1_in  = raw_dir  / f"{srr_id}_1.fastq.gz"
    r2_in  = raw_dir  / f"{srr_id}_2.fastq.gz"
    r1_out = out_dir  / f"{srr_id}_1.trimmed.fastq.gz"
    r2_out = out_dir  / f"{srr_id}_2.trimmed.fastq.gz"
    json_out = report_dir / f"{srr_id}_fastp.json"
    html_out = report_dir / f"{srr_id}_fastp.html"

    cmd = [
        "fastp",
        "--in1",          str(r1_in),
        "--in2",          str(r2_in),
        "--out1",         str(r1_out),
        "--out2",         str(r2_out),
        "--json",         str(json_out),
        "--html",         str(html_out),

        # ── Quality trimming ──────────────────────────────────────────────
        "--cut_right",             # sliding window from 3' end
        "--cut_window_size", "4",
        "--cut_mean_quality", "20",

        # ── Length filter ─────────────────────────────────────────────────
        "--length_required", "50",

        # ── Misc ──────────────────────────────────────────────────────────
        "--correction",            # overlap-based error correction
        "--detect_adapter_for_pe", # auto-detect adapters
        "--thread", str(threads),
        "--report_title", f"{srr_id} fastp Report",
    ]

    print(f"  Running fastp for {srr_id} ...", end="", flush=True)
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(" FAILED")
        print(result.stderr[-1500:])
        raise RuntimeError(f"fastp failed for {srr_id}")
    print(" done")

    # Parse JSON report for summary metrics
    with open(json_out) as fh:
        report = json.load(fh)

    before = report["summary"]["before_filtering"]
    after  = report["summary"]["after_filtering"]
    filt   = report["filtering_result"]

    return {
        "sample_id"           : srr_id,
        "label"               : SAMPLES[srr_id]["label"],
        "group"               : SAMPLES[srr_id]["group"],
        "reads_before_M"      : round(before["total_reads"] / 1e6, 2),
        "reads_after_M"       : round(after["total_reads"]  / 1e6, 2),
        "pct_reads_kept"      : round(100 * after["total_reads"] / before["total_reads"], 1),
        "q30_before"          : round(before["q30_rate"] * 100, 1),
        "q30_after"           : round(after["q30_rate"]  * 100, 1),
        "gc_before"           : round(before["gc_content"] * 100, 1),
        "gc_after"            : round(after["gc_content"]  * 100, 1),
        "reads_low_quality"   : filt["low_quality_reads"],
        "reads_too_short"     : filt["too_short_reads"],
        "reads_with_adapter"  : report["adapter_cutting"]["adapter_trimmed_reads"],
    }


# Run fastp for all samples
print("Running fastp trimming for all samples...\n")
fastp_metrics = []
for srr_id in SAMPLES:
    m = run_fastp(srr_id, RAW_DIR, TRIM_DIR, FASTP_DIR, THREADS)
    fastp_metrics.append(m)

df_fastp = pd.DataFrame(fastp_metrics)
print("\nfastp Trimming Summary:")
print(df_fastp.to_string(index=False))
df_fastp.to_csv(RESULTS_DIR / "fastp_summary.csv", index=False)

---
## Step 5 — Post-Trimming FastQC/MultiQC & Before/After Comparison

In [ ]:
# Run FastQC on trimmed reads
run_fastqc(TRIM_DIR, QC_TRIM_DIR, THREADS)

# Run MultiQC on trimmed reads (include fastp JSON reports too)
# We search both QC_TRIM_DIR and FASTP_DIR so MultiQC picks up fastp stats
combined_search = BASE_DIR / "qc"
run_multiqc(combined_search, MULTIQC_TRIM, report_name="multiqc_trimmed")

# Parse trimmed FastQC data
df_summary_trim = parse_fastqc_summaries(QC_TRIM_DIR)
df_metrics_trim = parse_fastqc_data(QC_TRIM_DIR)
df_flags_trim   = apply_qc_flags(df_metrics_trim)
print("\nPost-trim QC Flag Table:")
print(df_flags_trim[flag_cols].to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 2 – Post-trimming FastQC Heatmap
# ─────────────────────────────────────────────────────────────────────────────
plot_fastqc_heatmap(
    df_summary_trim,
    title="FastQC Module Status — Post-fastp Trimmed Reads",
    save_path=FIG_DIR / "fig2_fastqc_heatmap_trimmed.png"
)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 3 – Before vs After Comparison Bar Charts
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

labels    = df_fastp["label"].tolist()
x         = np.arange(len(labels))
bar_w     = 0.35

colors_grp = [HC_COLOR if g == "HC" else SS_COLOR for g in df_fastp["group"]]

# ── Panel A: Reads Before vs After ────────────────────────────────────────
ax = axes[0]
ax.bar(x - bar_w/2, df_fastp["reads_before_M"], bar_w, label="Before",
       color=[c + "99" for c in [HC_COLOR]*3 + [SS_COLOR]*3], edgecolor="grey")
ax.bar(x + bar_w/2, df_fastp["reads_after_M"],  bar_w, label="After",
       color=[HC_COLOR]*3 + [SS_COLOR]*3, edgecolor="grey")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Reads (millions)"); ax.set_title("A. Total Reads", fontweight="bold")
ax.legend(fontsize=8)

# ── Panel B: Q30 Rate Before vs After ─────────────────────────────────────
ax = axes[1]
ax.bar(x - bar_w/2, df_fastp["q30_before"], bar_w, label="Before",
       color="#aecde8", edgecolor="grey")
ax.bar(x + bar_w/2, df_fastp["q30_after"],  bar_w, label="After",
       color="#2166ac", edgecolor="grey")
ax.axhline(80, color="red", linestyle="--", linewidth=0.8, label="80% threshold")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("Q30 Rate (%)"); ax.set_title("B. Q30 Rate", fontweight="bold")
ax.legend(fontsize=8)

# ── Panel C: GC Content Before vs After ───────────────────────────────────
ax = axes[2]
ax.bar(x - bar_w/2, df_fastp["gc_before"], bar_w, label="Before",
       color="#a1d99b", edgecolor="grey")
ax.bar(x + bar_w/2, df_fastp["gc_after"],  bar_w, label="After",
       color="#31a354", edgecolor="grey")
ax.axhline(EXPECTED_GC, color="navy", linestyle="--", linewidth=0.8, label="Expected 50%")
ax.axhspan(EXPECTED_GC - GC_DEVIATION, EXPECTED_GC + GC_DEVIATION,
           alpha=0.08, color="navy", label="±10% band")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=8)
ax.set_ylabel("GC Content (%)"); ax.set_title("C. GC Content", fontweight="bold")
ax.legend(fontsize=8)

# Add group labels
for ax in axes:
    ax.axvline(2.5, color="black", linestyle=":", linewidth=0.8)
    ax.text(1, ax.get_ylim()[1]*0.95, "HC", ha="center", fontsize=9,
            color=HC_COLOR, fontweight="bold")
    ax.text(4, ax.get_ylim()[1]*0.95, "SS", ha="center", fontsize=9,
            color=SS_COLOR, fontweight="bold")

fig.suptitle("Before vs After fastp Trimming — PBMC RNA-seq (GSE96960)",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(FIG_DIR / "fig3_before_after_trimming.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {FIG_DIR / 'fig3_before_after_trimming.png'}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Deliverable — Final Summary Table
# Pre/post-trimming QC flag summary table per sample
# ─────────────────────────────────────────────────────────────────────────────

# Merge raw and trimmed metrics (mean across R1+R2 per sample for display)
def mean_per_sample(df: pd.DataFrame, suffix: str) -> pd.DataFrame:
    grp = df.groupby(["sample_id", "label", "group"]).agg(
        gc_pct        = ("gc_pct",         "mean"),
        min_mean_q    = ("min_mean_q",      "mean"),
        max_adapter   = ("max_adapter_pct", "max"),
        total_reads_M = ("total_reads_M",   "mean"),
    ).reset_index()
    grp.columns = [
        "sample_id", "label", "group",
        f"gc_pct_{suffix}",
        f"min_q_{suffix}",
        f"adapter_{suffix}",
        f"reads_M_{suffix}",
    ]
    return grp

raw_agg  = mean_per_sample(df_metrics_raw,  "raw")
trim_agg = mean_per_sample(df_metrics_trim, "trim")

summary_table = raw_agg.merge(trim_agg[["sample_id", "gc_pct_trim", "min_q_trim",
                                         "adapter_trim", "reads_M_trim"]],
                               on="sample_id")

# Add fastp stats
summary_table = summary_table.merge(
    df_fastp[["sample_id", "pct_reads_kept", "q30_before", "q30_after"]],
    on="sample_id"
)

# QC pass/fail per stage
summary_table["raw_flag"]  = summary_table.apply(
    lambda r: "FAIL" if (r.min_q_raw < 20 or r.adapter_raw > 5 or
                         abs(r.gc_pct_raw - 50) > 10) else "PASS", axis=1)
summary_table["trim_flag"] = summary_table.apply(
    lambda r: "FAIL" if (r.min_q_trim < 20 or r.adapter_trim > 5 or
                         abs(r.gc_pct_trim - 50) > 10) else "PASS", axis=1)

print("\n" + "="*80)
print("FINAL QC SUMMARY TABLE — Pre/Post Trimming")
print("="*80)
display_cols = [
    "label", "group",
    "reads_M_raw", "gc_pct_raw", "min_q_raw", "adapter_raw", "raw_flag",
    "reads_M_trim", "gc_pct_trim", "min_q_trim", "adapter_trim", "trim_flag",
    "pct_reads_kept", "q30_before", "q30_after",
]
print(summary_table[display_cols].round(2).to_string(index=False))

summary_table[display_cols].round(2).to_csv(
    RESULTS_DIR / "summary_table_pre_post_trimming.csv", index=False)
print(f"\n✓ Final summary saved: {RESULTS_DIR / 'summary_table_pre_post_trimming.csv'}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4 – PASS/FAIL module improvement heatmap (raw vs trimmed)
# ─────────────────────────────────────────────────────────────────────────────
STATUS_MAP2 = {"PASS": 0, "WARN": 1, "FAIL": 2}

def module_change_summary(df_before: pd.DataFrame, df_after: pd.DataFrame) -> pd.DataFrame:
    """Count PASS/WARN/FAIL per module before and after trimming."""
    before_counts = (
        df_before.groupby(["module", "status"])
        .size().unstack(fill_value=0)
        .add_suffix("_raw")
    )
    after_counts = (
        df_after.groupby(["module", "status"])
        .size().unstack(fill_value=0)
        .add_suffix("_trim")
    )
    combined = before_counts.join(after_counts, how="outer").fillna(0).astype(int)
    return combined

change_df = module_change_summary(df_summary_raw, df_summary_trim)
print("Module Status Changes (raw → trimmed):")
print(change_df.to_string())
change_df.to_csv(RESULTS_DIR / "module_status_change.csv")
print(f"\n✓ Saved: {RESULTS_DIR / 'module_status_change.csv'}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4 – Adapter content change across modules
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df_s, stage in zip(axes,
                            [df_summary_raw, df_summary_trim],
                            ["Raw", "Trimmed"]):
    df_s2 = df_s.copy()
    df_s2["col"]   = df_s2["label"] + "_" + df_s2["read"]
    df_s2["status_n"] = df_s2["status"].map(STATUS_MAP2)
    pivot = df_s2.pivot_table(
        index="module", columns="col",
        values="status_n", aggfunc="first"
    )
    cmap = plt.cm.colors.ListedColormap([PASS_COLOR, WARN_COLOR, FAIL_COLOR])
    ax.imshow(pivot.values.astype(float), cmap=cmap, vmin=0, vmax=2, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=7)
    ax.set_title(f"{stage} Reads", fontweight="bold")

patches = [mpatches.Patch(facecolor=PASS_COLOR, label="PASS"),
           mpatches.Patch(facecolor=WARN_COLOR, label="WARN"),
           mpatches.Patch(facecolor=FAIL_COLOR, label="FAIL")]
axes[1].legend(handles=patches, loc="upper right",
               bbox_to_anchor=(1.25, 1), fontsize=8)

fig.suptitle("FastQC Module Status: Raw vs Trimmed",
             fontsize=12, fontweight="bold")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig4_raw_vs_trimmed_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {FIG_DIR / 'fig4_raw_vs_trimmed_heatmap.png'}")

---
## Trimming Parameter Justification

| Parameter | Value | Rationale |
|---|---|---|
| `--cut_right` | sliding window 3'→5' | Illumina quality degrades at 3' ends; window trimming is gentler than hard trimming |
| `--cut_window_size 4` | 4 bp | Recommended by fastp docs; balances sensitivity vs specificity |
| `--cut_mean_quality 20` | Q20 | Standard threshold; Q20 = 99% base call accuracy |
| `--length_required 50` | 50 bp | Reads < 50 bp map unreliably for RNA-seq; ENCODE standard |
| `--detect_adapter_for_pe` | auto | fastp detects adapters from read overlap; no manual adapter sequence needed |
| `--correction` | enabled | Overlap-based correction reduces mismatch errors in paired-end reads |

**Why not Trimmomatic?** fastp is 3–8× faster (SIMD-accelerated), produces cleaner JSON reports for MultiQC, and its auto adapter detection outperforms Trimmomatic's fixed adapter lists on modern Illumina library preps (Chen et al., 2018 *Bioinformatics*).

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Final Deliverables Summary
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 65)
print(" DAY 01 DELIVERABLES CHECKLIST")
print("=" * 65)

deliverables = [
    (MULTIQC_RAW  / "multiqc_raw.html",       "Pre-trim MultiQC HTML report"),
    (MULTIQC_TRIM / "multiqc_trimmed.html",   "Post-trim MultiQC HTML report"),
    (RESULTS_DIR  / "qc_flags_raw.csv",        "QC flag table (raw reads)"),
    (RESULTS_DIR  / "fastp_summary.csv",       "fastp trimming metrics per sample"),
    (RESULTS_DIR  / "summary_table_pre_post_trimming.csv", "Pre/post comparison table"),
    (RESULTS_DIR  / "module_status_change.csv","Module status change summary"),
    (FIG_DIR      / "fig1_fastqc_heatmap_raw.png",     "Fig 1: Raw FastQC heatmap"),
    (FIG_DIR      / "fig2_fastqc_heatmap_trimmed.png", "Fig 2: Trimmed FastQC heatmap"),
    (FIG_DIR      / "fig3_before_after_trimming.png",  "Fig 3: Reads/Q30/GC comparison"),
    (FIG_DIR      / "fig4_raw_vs_trimmed_heatmap.png", "Fig 4: Raw vs trimmed side-by-side"),
]

for path, desc in deliverables:
    status = "✓" if path.exists() else "✗ MISSING"
    size   = f"({path.stat().st_size/1024:.1f} KB)" if path.exists() else ""
    print(f"  {status}  {desc}\n     └─ {path.name} {size}")

print("\nKey Paper:")
print("  Ewels et al. (2016) 'MultiQC: summarize analysis results for")
print("  multiple tools and samples in a single report' — Bioinformatics")
print("  Chen et al. (2018) 'fastp: an ultra-fast all-in-one FASTQ preprocessor'")
print("  — Bioinformatics")